# Polynomial Regression

So far our models have been **straight lines** (or planes, when we had several features). But many relationships in the real world are curved: the braking distance of a car grows much faster than its speed, fuel consumption goes up both at very low and very high speeds, and so on.

The good news is that we do **not** need a new algorithm to model curves. Polynomial regression is *linear regression applied to new features* built from the original one:

$$\hat{y} = w_0 + w_1 x + w_2 x^2 + \dots + w_d x^d$$

The model is non-linear in $x$ but still **linear in the weights** $w_j$, so everything we have learnt (least squares, gradient descent, $R^2$...) still applies.

**In this lab you will:**
1. See why a straight line is not enough for curved data.
2. Build polynomial features and fit models of different degrees.
3. Choose the degree with a validation set, and see under- and overfitting in action.
4. Understand why we normalise the features before running gradient descent.
5. See why extrapolating with high-degree polynomials is dangerous.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4.5)

## 1. The data: braking distance

We will use a **synthetic** dataset (generated by us, so we know what is going on behind the scenes) inspired by a real physical law: the braking distance of a car grows with the *square* of its speed. We add random noise to simulate measurement errors.

In [ ]:
rng = np.random.default_rng(1)
n = 45
speed = rng.uniform(10, 120, n)                                       # km/h
distance = 0.0075 * speed**2 + 0.15 * speed + rng.normal(0, 6, n)     # metres, with noise

plt.scatter(speed, distance, edgecolor="k")
plt.xlabel("Speed (km/h)")
plt.ylabel("Braking distance (m)")
plt.show()

We split the data into a **training set** (25 points), used to fit the models, and a **validation set** (20 points), kept aside to measure how well each model works on data it has not seen. We will need it later to choose the degree of the polynomial.

In [ ]:
perm = np.random.default_rng(0).permutation(n)
train, val = perm[:25], perm[25:]
x_train, y_train = speed[train], distance[train]
x_val, y_val = speed[val], distance[val]

print("Training points:", len(x_train), "| Validation points:", len(x_val))

## 2. A straight line first (baseline)

Let's start with what we already know: $\hat{y} = w_0 + w_1 x$. We build the *design matrix* $X$ (a column of ones for the bias and a column with $x$) and solve the least squares problem with `np.linalg.lstsq`.

We also define the **RMSE** (root mean squared error), which measures the typical error in the same units as $y$ (metres here).

In [ ]:
def rmse(y, y_pred):
    return np.sqrt(np.mean((y - y_pred) ** 2))


X_lin = np.column_stack([np.ones_like(x_train), x_train])
w_lin, *_ = np.linalg.lstsq(X_lin, y_train, rcond=None)

print(f"y_hat = {w_lin[0]:.2f} + {w_lin[1]:.3f} * x")
print("Training RMSE:", round(rmse(y_train, X_lin @ w_lin), 2), "m")

In [ ]:
xs = np.linspace(0, 130, 200)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].scatter(x_train, y_train, edgecolor="k", label="Training data")
ax[0].plot(xs, w_lin[0] + w_lin[1] * xs, color="red", label="Straight line")
ax[0].set_xlabel("Speed (km/h)")
ax[0].set_ylabel("Braking distance (m)")
ax[0].set_title("Linear fit")
ax[0].legend()

residuals = y_train - X_lin @ w_lin
ax[1].scatter(x_train, residuals, edgecolor="k")
ax[1].axhline(0, color="red")
ax[1].set_xlabel("Speed (km/h)")
ax[1].set_ylabel("Residual (m)")
ax[1].set_title("Residuals of the linear fit")
plt.show()

The line misses the data in a **systematic** way: it is too high for medium speeds and too low for the extremes. That pattern in the residuals (a "U" shape instead of random noise around zero) is the sign that the model is too simple: it **underfits**.

## 3. Polynomial features

To fit a curve we give the model extra columns: $x^2, x^3, \dots, x^d$. The function below builds the design matrix

$$X = \begin{bmatrix} 1 & z_1 & z_1^2 & \dots & z_1^d \\ \vdots & & & & \vdots \\ 1 & z_m & z_m^2 & \dots & z_m^d \end{bmatrix}$$

where $z$ is the **standardised** input, $z = (x - \mu)/\sigma$. We standardise *before* taking powers so that all the columns have a reasonable size (section 5 shows what happens if we don't). The mean $\mu$ and the standard deviation $\sigma$ are computed on the **training set only** and then reused for any new data.

In [ ]:
def map_feature(x, degree, mu=None, sigma=None):
    """Design matrix [1, z, z^2, ..., z^degree] with z = (x - mu) / sigma.
    If mu and sigma are not given they are computed from x (use the training set!)."""
    if mu is None:
        mu, sigma = x.mean(), x.std()
    z = (x - mu) / sigma
    X = np.column_stack([z ** k for k in range(degree + 1)])
    return X, mu, sigma


X_example, mu_ex, sigma_ex = map_feature(np.array([20.0, 60.0, 100.0]), degree=3)
print(np.round(X_example, 2))

Because the model is linear in the weights, fitting it is exactly the same problem as before: least squares on the new design matrix. We wrap fitting and prediction in two small functions. A `model` is just the tuple `(w, mu, sigma)`.

In [ ]:
def fit_poly(x, y, degree):
    X, mu, sigma = map_feature(x, degree)
    w, *_ = np.linalg.lstsq(X, y, rcond=None)
    return w, mu, sigma


def predict_poly(x, model):
    w, mu, sigma = model
    X, _, _ = map_feature(x, len(w) - 1, mu, sigma)
    return X @ w

Let's fit polynomials of increasing degree. **Blue** points are the training data (used to fit), **orange** points are validation data (never seen by the model). Each title shows the RMSE on the training and on the validation set.

In [ ]:
degrees = [1, 2, 3, 6, 10]
xs = np.linspace(x_train.min() - 5, x_train.max() + 5, 300)

fig, axes = plt.subplots(1, len(degrees), sharey=True, figsize=(20, 4))
for degree, ax in zip(degrees, axes):
    model = fit_poly(x_train, y_train, degree)
    ax.scatter(x_train, y_train, edgecolor="k", label="train")
    ax.scatter(x_val, y_val, edgecolor="k", color="orange", label="validation")
    ax.plot(xs, predict_poly(xs, model), color="red")
    ax.set_ylim(distance.min() - 20, distance.max() + 40)
    ax.set_xlabel("Speed (km/h)")
    ax.set_title(f"Degree {degree}\ntrain {rmse(y_train, predict_poly(x_train, model)):.1f} | "
                 f"val {rmse(y_val, predict_poly(x_val, model)):.1f}")
axes[0].set_ylabel("Braking distance (m)")
axes[0].legend()
plt.show()

What do we see?

* **Degree 1** is too rigid: high error on both sets (*underfitting*).
* **Degrees 2-3** follow the trend: low error on both sets.
* **Degrees 6 and 10** pass very close to the training points but oscillate wildly between them, and the validation error explodes (*overfitting*). The model has started to memorise the noise.

## 4. Choosing the degree

The training error can only go **down** as we add degrees (a more flexible model can always fit the training data at least as well), so it is useless for choosing the degree. The **validation error** is what tells us how well the model generalises. Let's compute both for every degree from 1 to 10.

In [ ]:
degrees = list(range(1, 11))
train_err, val_err = [], []
for degree in degrees:
    model = fit_poly(x_train, y_train, degree)
    train_err.append(rmse(y_train, predict_poly(x_train, model)))
    val_err.append(rmse(y_val, predict_poly(x_val, model)))

for degree, tr, va in zip(degrees, train_err, val_err):
    print(f"degree {degree:2d}: train RMSE = {tr:6.2f} | validation RMSE = {va:7.2f}")

plt.semilogy(degrees, train_err, "o-", label="Training")
plt.semilogy(degrees, val_err, "o-", label="Validation")
plt.xlabel("Degree of the polynomial")
plt.ylabel("RMSE (m, log scale)")
plt.legend()
plt.show()

best_degree = degrees[int(np.argmin(val_err))]
print("Best degree according to the validation set:", best_degree)

The training curve keeps falling while the validation curve has a **minimum** and then shoots up: the classic signature of overfitting. We pick the degree that minimises the validation error.

> **Caution:** with only 20 validation points this estimate is noisy; if you change the random split you may get degree 2 instead of 3. In the *Regularization* lab we will see a more robust way of choosing hyperparameters (cross-validation) and a way of keeping a high degree without overfitting.

## 5. Gradient descent, and why we normalise

`lstsq` gives the exact solution in one shot, but in this course we have learnt to train models with **gradient descent**, which is what we will use for models without a closed-form solution (like logistic regression). Here is the same function we have been using:

In [ ]:
def gradient_descent(X, y, alpha, iterations):
    m = len(y)
    w = np.zeros(X.shape[1])
    cost = []
    for _ in range(iterations):
        error = X @ w - y
        cost.append(error @ error / (2 * m))
        w = w - alpha * (X.T @ error) / m
    return w, np.array(cost)

First, what happens if we use the **raw** powers of the speed ($1, x, x^2, x^3$) without normalising? Look at the size of the columns:

In [ ]:
X_raw = np.column_stack([x_train ** k for k in range(4)])
print("Largest value of each column:", X_raw.max(axis=0))

with np.errstate(all="ignore"):
    w_a, cost_a = gradient_descent(X_raw, y_train, alpha=1e-12, iterations=2000)
    w_b, cost_b = gradient_descent(X_raw, y_train, alpha=1e-9, iterations=2000)

print("alpha = 1e-12  -> final cost:", cost_a[-1])
print("alpha = 1e-9   -> final cost:", cost_b[-1])

The columns differ by **six orders of magnitude**. A learning rate big enough to make progress on the small columns (the bias, $x$) makes the weight of the huge column ($x^3$) jump back and forth and diverge (`nan`), while a learning rate small enough to be stable barely moves the other weights in 2000 iterations.

With the standardised features, all columns are of order 1, and a single learning rate works for all of them:

In [ ]:
X_std, mu, sigma = map_feature(x_train, degree=3)
w_gd, cost_gd = gradient_descent(X_std, y_train, alpha=0.1, iterations=2000)

w_exact, *_ = np.linalg.lstsq(X_std, y_train, rcond=None)
print("Largest difference between gradient descent and lstsq weights:", np.abs(w_gd - w_exact).max())

plt.semilogy(cost_a, label="Raw features (alpha = 1e-12)")
plt.semilogy(cost_gd, label="Standardised features (alpha = 0.1)")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.legend()
plt.show()

Gradient descent on the standardised features reaches **exactly** the least squares solution within 2000 iterations (the largest weight difference printed above is tiny), while the raw features are still far from it. This is why we always normalise polynomial features.

## 6. Beware of extrapolation

A fitted polynomial is only trustworthy **inside the range of the training data** (10-120 km/h here). What would each model predict for a car driving at 150 km/h?

In [ ]:
for degree in [1, 2, 3, 10]:
    model = fit_poly(x_train, y_train, degree)
    prediction = predict_poly(np.array([150.0]), model)[0]
    print(f"degree {degree:2d}: predicted braking distance at 150 km/h = {prediction:10.1f} m")

The true value for this synthetic data is about 190 m. Degrees 2 and 3 get close, the straight line underestimates it, and the degree-10 polynomial predicts a *negative* distance of tens of kilometres. High-degree polynomials shoot off to $\pm\infty$ very quickly outside the data, so **never extrapolate** with them.

## 7. In practice

You do not need to write all of this yourself every time:

* `np.polyfit(x, y, degree)` fits a polynomial in one line (it returns the coefficients, highest power first) and `np.polyval` evaluates it.
* scikit-learn offers `PolynomialFeatures` (to build the features), `StandardScaler` (to normalise) and `LinearRegression`, which can be chained with `make_pipeline`.

Let's check that `np.polyfit` agrees with our implementation:

In [ ]:
coeffs = np.polyfit(x_train, y_train, 3)
model_3 = fit_poly(x_train, y_train, 3)

print("Same predictions on the validation set:",
      np.allclose(np.polyval(coeffs, x_val), predict_poly(x_val, model_3)))

## Exercises

### Exercise 1: R²

Write a function `r2_score(y, y_pred)` that computes the coefficient of determination

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

Then use it to compute $R^2$ of the degree-1 model and of the degree-3 model on the **validation** set. What do you conclude?

In [ ]:
# Your Code Here

---

*Your Answer Here*

---

### Exercise 2: fuel consumption

The car's fuel consumption (litres per 100 km) is high at very low and at very high speeds, and lowest at a moderate speed. The following (synthetic) data simulates 40 measurements:

In [ ]:
rng2 = np.random.default_rng(5)
speed2 = rng2.uniform(30, 160, 40)                                  # km/h
fuel = 5 + 0.0008 * (speed2 - 85) ** 2 + rng2.normal(0, 0.35, 40)   # L/100km, with noise

1. Plot the data.
2. Split it randomly into a training set (25 points) and a validation set (15 points).
3. Fit polynomials of degree 1 to 8 on the training set and compute the validation RMSE of each one.
4. Which degree would you choose? Plot the chosen model on top of the data.

*Hint: reuse `fit_poly`, `predict_poly` and `rmse`.*

In [ ]:
# Your Code Here

---

*Your Answer Here*

---

### Exercise 3: a suspicious report

A colleague fits a degree-10 polynomial to 25 data points and reports: *"My model has a training RMSE of almost 0, so it is the best possible model."* Do you agree? Explain what could be wrong and what you would check.

---

*Your Answer Here*

---